# Phase 1 — Global earthquake clustering
This notebook only clones/pulls the GitHub repo and runs the logic from `src/`.
To change the code, edit the files under `src/` on GitHub and re-run this notebook (the clone cell pulls automatically).

In [ ]:
import os

CONTENT_DIR = "/content"  # fixed, known root -- re-running this cell is always safe
REPO_URL = "https://github.com/AHK-0214/CI_Clustering.git"
REPO_DIR = os.path.join(CONTENT_DIR, "CI_Clustering")

os.chdir(CONTENT_DIR)  # undo any previous %cd before deciding anything

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print("Repo already exists; pulling latest changes...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Current directory:", os.getcwd())

# Force a fresh import of src/* in the cell below, even if this kernel already
# imported an older version of it earlier in this session. Without this, a
# `git pull` updates the files on disk but Python keeps serving the OLD cached
# module from sys.modules, which is the classic cause of a stale
# `KeyError: 'p1_structures'` (or any other "it's right there in the code but
# still fails") after pulling new code without restarting the runtime.
import sys
for _mod_name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_mod_name]


In [ ]:
!pip install -q -r requirements.txt

## Dataset
Pick one of two options:

**Option 1 (recommended, simplest):** download `database.csv` from Kaggle and upload it here.

**Option 2:** upload your own `kaggle.json` so kagglehub can download it automatically.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

USE_KAGGLEHUB = False  # set to True to try automatic download via kagglehub

if USE_KAGGLEHUB:
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        os.rename(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
else:
    if not os.path.exists("data/database.csv"):
        from google.colab import files
        print("Upload the database.csv file (from the usgs/earthquake-database Kaggle dataset):")
        uploaded = files.upload()
        for fname in uploaded:
            os.rename(fname, "data/database.csv")
    else:
        print("data/database.csv already exists.")

In [ ]:
import sys, importlib
sys.path.insert(0, ".")

# Import (or re-import) the current on-disk version of src.phase1 -- see the
# cache-busting note in the clone/pull cell above.
import src.phase1 as _phase1_mod
importlib.reload(_phase1_mod)
from src.phase1 import run_phase1

import matplotlib.pyplot as plt
plt.ioff()  # run_phase1() builds several matplotlib figures; ioff() plus the
            # plt.close(fig) calls inside src/viz.py stop them from being
            # auto-printed here a second time -- each figure is only shown once,
            # later, in its own cell under Questions A/B/D below.
results = run_phase1(save_outputs=True)
plt.ion()

print("Total earthquakes after cleaning (Type == Earthquake):", results["n_total"])
print("Base seed (s):", results["seed_base"])


## Question A — map and three regions

In [ ]:
results["figures"]["map"]

Real-world map with country borders/coastlines (hover to see lat/lon), so each earthquake can be tied to an actual country or region instead of a bare scatter plot:

In [ ]:
results["figures"]["map_geo"]

**Note:** the region near ±180° longitude (e.g. the Tonga-Kermadec trench) may look split into two pieces at the left/right edge of the flat map above, because any flat map of a sphere necessarily has a "seam" somewhere. This is only a 2D-display limitation; the distance calculations (conversion to xyz on the unit sphere in `src/geo.py`) do not have this problem, and points on either side of this line are correctly treated as near each other. (Full explanation: Section C of the English report, `report_phase1_en.docx`.)


In [ ]:
results["figures"]["map_regions"]

In [ ]:
results["region_table"]

## Question B — density (distance to the 10th neighbor)

In [ ]:
results["figures"]["hist"]

In [ ]:
results["density_table"]

## Question C — why plain Euclidean distance on lat/lon is misleading

In [ ]:
print(results["distance_explanation"])
print()
print("(Also written out in full in Section C of the English report, report_phase1_en.docx.)")


## Question D — predictions P1 through P4

In [ ]:
print("Predictions P1-P4 (prediction, reason, rejection criterion) are also written out "
      "in full in Section D of the English report (report_phase1_en.docx). The table below "
      "shows the same predictions dict used to generate that section.")
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.DataFrame(results["predictions"]).T


### Question D (continued) — where are the structures named in P1?
This is a subsection of Question D / prediction P1 above, not a separate question. Still raw data (no clustering algorithm has been run) — just locating the candidate elongated structures mentioned in P1's reasoning (Mid-Atlantic Ridge, South America's west coast, the East Asia / western Pacific arc) on the map.


In [ ]:
results["figures"]["p1_structures"]


## Download outputs (to attach to the report)
All files are saved under `outputs/phase1/`; the cell below zips and downloads all of them.

In [ ]:
!zip -r phase1_outputs.zip outputs/phase1
from google.colab import files
files.download("phase1_outputs.zip")